# U11 | Bahdanau Attention

**目标**：在 U10 翻译 Baseline 的基础上，加入 Bahdanau Attention，让 Decoder 每一步都能「回头看」整个源句，而不是只啃一个固定的 context vector。

**过关标准**：
- 模型在 U10 同样的小语料上能跑通，loss 收敛到 0.1 附近
- 能画出注意力热力图（attention heatmap），观察对齐关系
- 能默写四步公式：score → softmax → 加权求和 → 拼接进 Decoder

**前置依赖**：
- U08：Seq2Seq + Teacher Forcing
- U10：翻译 Baseline、`pack_padded_sequence`、`ignore_index=PAD`

## 本单元会回答
1. 为什么 U10 Baseline 不够？信息瓶颈具体是什么？
2. Bahdanau Attention 的四步计算：score / softmax / context / 拼接
3. Encoder 怎么改：从只返回 hidden 到返回每个时间步的 outputs
4. Decoder 怎么改：每步先算 attention，再喂 GRU
5. 怎么处理 pad 位置（mask）才不污染 softmax
6. 怎么收集并可视化注意力权重


## 1. 回顾：U10 Baseline 的硬伤

```
Encoder: src → GRU → hidden(1, B, H)   ← 整句压成一个固定向量
Decoder: 每步都用同一个 hidden 作为初始状态
```

三个问题：
1. **长句信息丢失**：3 字句 vs 30 字句都压成同样大小的 H 维向量
2. **Decoder 看不到细节**：所有步共享同一个 context，无法定位到源句某个具体位置
3. **对齐能力差**：翻译有天然的对齐关系（「我爱你」的「你」对应 `you`），固定 context 几乎学不到

**核心症结**：信息从 Encoder 流向 Decoder，只有一根「水管」（最后一步 hidden）。


## 2. Attention 的核心思想：动态 context

让 Decoder **每一步都重新看一遍源句**：

```
Baseline:   Decoder 每步都用 同一个 c
Attention:  Decoder 每步算  c_t = Σ α_tj · h_j   ← 不同 t 不同 c
                              ↑
                        attention 权重，softmax 出来的
```

**关键变化**：
- Encoder **保留全部时间步的 outputs**，形状 `(B, T_src, H)`
- Decoder 每步通过一个小网络给源句每个位置打分，再 softmax 成权重 `α`，加权求和得 `c_t`
- `c_t` 跟 Decoder 当前的输入拼起来再喂 GRU

直觉：Decoder 每生成一个目标词，就**问自己一句**——「我现在要用源句的哪些位置？」答案以权重分布的形式给出，并据此动态合成 context。


## 3. Bahdanau Attention 公式

### 3.1 符号约定

| 符号 | 含义 | 形状 |
|---|---|---|
| `s_{t-1}` | Decoder 上一步隐藏态 | `(B, H)` |
| `h_j`     | Encoder 第 j 位置的输出 | `(B, H)`（j ∈ 1..T_src） |
| `e_{tj}`  | 位置 j 对当前 t 步的「分数」 | `(B,)` |
| `α_{tj}`  | softmax 后的注意力权重 | `(B,)` |
| `c_t`     | 加权求和后的 context vector | `(B, H)` |

### 3.2 四步计算（每步标 shape）

**Step 1：算分数（score）**

$$e_{tj} = v^\top \cdot \tanh(W_a \cdot h_j + U_a \cdot s_{t-1})$$

- `W_a, U_a`：两个线性层，把 `h_j` 和 `s_{t-1}` 投到同维度
- `v`：把投影后的向量压成一个标量分数
- 实际实现一次性对所有 j 并行算，结果形状 `(B, T_src)`

**Step 2：归一化为权重（softmax）**

$$\alpha_{tj} = \frac{\exp(e_{tj})}{\sum_{k} \exp(e_{tk})}$$

- 在 `T_src` 维上做 softmax，结果形状 `(B, T_src)`
- 每行加起来 = 1，是一个分布

**Step 3：加权求和（context）**

$$c_t = \sum_{j=1}^{T_{src}} \alpha_{tj} \cdot h_j$$

- 形状 `(B, H)`，跟 baseline 的 hidden 一样
- 但**每个 t 都不一样**，这是关键

**Step 4：拼接喂给 Decoder GRU**

$$s_t = \mathrm{GRU}([\,e(y_{t-1});\, c_t\,],\; s_{t-1})$$

- `e(y_{t-1})` 是上一步目标词的 embedding，形状 `(B, E)`
- 跟 `c_t` 沿最后一维拼接 → `(B, E+H)`
- 喂进 GRU 得到新 hidden `s_t`，再过 Linear 得到 logits

### 3.3 为什么是 Bahdanau？

Bahdanau et al., 2014 提出的「**加性 attention**」：score 用 `v · tanh(W·h + U·s)` 这种**加法**形式。
后来 Luong 2015 提出「**乘性 attention**」：`score = h^T · W · s`，更简洁。
Transformer 的 scaled dot-product 是乘性 attention 的进一步简化。
本单元只实现 Bahdanau，理解了它，剩下的都好说。


## 4. 改造 Encoder：返回每一步的 outputs

U10 里只用 `hidden`：

```python
_, hidden = self.gru(packed)
return hidden          # (1, B, H)
```

U11 必须把 `outputs`（每个时间步的 hidden）也吐出来，给 Attention 用：

```python
out_packed, hidden = self.gru(packed)
outputs, _ = pad_packed_sequence(out_packed, batch_first=True)
return outputs, hidden  # outputs: (B, T_src, H)；hidden: (1, B, H)
```

**注意**：`outputs` 的 pad 位置是用 0 填充的（PyTorch 还原时默认就是 0），但**这些 0 不能直接进 softmax**——会被分配非零权重，污染结果。所以下一节要专门讲 mask。


## 5. 处理 pad：attention mask

源句末尾的 pad 位置在 outputs 里是 0 向量，但 score 函数可能给它一个非零分数 `e_{tj}`。
softmax 会把它的权重也算进去——本来应该是 0 的位置占了概率，**剩下真实 token 的权重就被稀释**了。

**解决方法**：在 softmax 之前，把 pad 位置的 score 设成 `-inf`：

```python
# scores: (B, T_src)
mask = (src != PAD)              # (B, T_src) 的布尔，True 表示真实 token
scores = scores.masked_fill(~mask, float('-inf'))
alpha = F.softmax(scores, dim=-1)  # pad 位置自然变 0
```

**为什么是 `-inf`？** 因为 `exp(-inf) = 0`，softmax 分子分母都干净，pad 位置权重精确为 0。

**`mask` 的 shape 与 src 一致**：`(B, T_src)`。我们已经在 collate 阶段把 `src` 传进了模型，`src == PAD` 即可拿到 mask，不必额外维护。


## 6. 改造 Decoder：每步先算 attention

新的 Decoder forward 单步流程：

```
输入：input_tok(B,1)、s_{t-1}(1,B,H)、enc_outputs(B,T_src,H)、src_mask(B,T_src)

1) embed = Embedding(input_tok)            # (B, 1, E)
2) score = v · tanh(W·enc + U·s)           # (B, T_src)
3) score = score.masked_fill(~mask, -inf)
4) alpha = softmax(score)                   # (B, T_src)
5) c_t   = (alpha · enc_outputs).sum(1)    # (B, H)
6) gru_in = cat(embed, c_t.unsqueeze(1), -1) # (B, 1, E+H)
7) out, s_t = GRU(gru_in, s_{t-1})         # out: (B, 1, H)
8) logits = Linear(out.squeeze(1))         # (B, V)
return logits, s_t, alpha
```

注意：
- 我们**返回了 `alpha`**，方便后面画热力图
- `nn.GRU(input_size=E+H, hidden_size=H)`：输入维度变成 `E+H`，输出还是 `H`
- 第一步的 `s_{t-1}` 来自 Encoder 最后一个 hidden（跟 baseline 一样）


## 7. 代码实现

下面把 U10 的代码搬过来微调：复用数据流水线，只替换 `Encoder` / `Decoder` / `Seq2Seq`。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ['<pad>', '<sos>', '<eos>', '<unk>']

EMBED_DIM  = 64
HIDDEN_DIM = 128
BATCH_SIZE = 14
LR = 1e-3
EPOCHS = 100

torch.manual_seed(42)
random.seed(42)


### 7.1 数据准备（与 U10 完全相同）


In [ ]:
raw_pairs = [
    ('我爱你', 'i love you'),
    ('我喜欢猫', 'i like cats'),
    ('他在看书', 'he is reading a book'),
    ('今天天气真好', 'the weather is nice today'),
    ('她正在学习深度学习', 'she is learning deep learning'),
    ('我们一起去公园', 'let us go to the park'),
    ('这本书很有趣', 'this book is interesting'),
    ('你叫什么名字', 'what is your name'),
    ('我来自中国', 'i am from china'),
    ('明天见', 'see you tomorrow'),
    ('我饿了', 'i am hungry'),
    ('他会说英语', 'he can speak english'),
    ('我想喝水', 'i want some water'),
    ('谢谢你', 'thank you'),
    ('对不起', 'i am sorry'),
]

def tokenize_zh(text): return list(text)
def tokenize_en(text): return text.lower().split()

class Vocab:
    def __init__(self, token_lists, min_freq=1):
        counter = Counter()
        for tokens in token_lists:
            counter.update(tokens)
        self.itos = list(SPECIALS) + [t for t, c in counter.most_common() if c >= min_freq]
        self.stoi = {t: i for i, t in enumerate(self.itos)}
    def __len__(self): return len(self.itos)
    def encode(self, tokens): return [self.stoi.get(t, UNK) for t in tokens]
    def decode(self, ids):    return [self.itos[i] for i in ids]

src_vocab = Vocab([tokenize_zh(zh) for zh, en in raw_pairs])
tgt_vocab = Vocab([tokenize_en(en) for zh, en in raw_pairs])

def sentence_to_ids(sentence, vocab, tokenizer, add_sos=False, add_eos=True):
    ids = vocab.encode(tokenizer(sentence))
    if add_sos: ids = [SOS] + ids
    if add_eos: ids = ids + [EOS]
    return ids

def pad_sequence(ids_list, pad_id=PAD):
    max_len = max(len(ids) for ids in ids_list)
    padded = [ids + [pad_id] * (max_len - len(ids)) for ids in ids_list]
    return torch.tensor(padded, dtype=torch.long)

class TranslationDataset(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self):         return len(self.pairs)
    def __getitem__(self, idx):
        zh, en = self.pairs[idx]
        src_ids = sentence_to_ids(zh, src_vocab, tokenize_zh, add_sos=False, add_eos=True)
        tgt_ids = sentence_to_ids(en, tgt_vocab, tokenize_en, add_sos=True,  add_eos=True)
        return src_ids, tgt_ids

def collate_fn(batch):
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    src_list, tgt_list = zip(*batch)
    src_len = torch.tensor([len(s) for s in src_list], dtype=torch.long)
    src = pad_sequence(list(src_list))
    tgt = pad_sequence(list(tgt_list))
    return src, src_len, tgt

dataset = TranslationDataset(raw_pairs)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
print(f'src_vocab={len(src_vocab)}, tgt_vocab={len(tgt_vocab)}')


### 7.2 Encoder：返回 outputs 和 hidden


In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, src, src_len):
        # src: (B, T_src)   src_len: (B,)
        embedded = self.embedding(src)                                       # (B, T, E)
        packed   = pack_padded_sequence(embedded, src_len.cpu(), batch_first=True)
        out_packed, hidden = self.gru(packed)
        outputs, _ = pad_packed_sequence(out_packed, batch_first=True)       # (B, T_src, H)
        return outputs, hidden                                                # outputs:(B,T,H); hidden:(1,B,H)

# 自测
enc = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM)
for src, src_len, tgt in loader:
    outs, h = enc(src, src_len)
    print('outputs:', outs.shape, '  hidden:', h.shape)
    break


### 7.3 BahdanauAttention 模块


In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim, bias=False)   # 投影 enc_outputs
        self.U = nn.Linear(hidden_dim, hidden_dim, bias=False)   # 投影 dec_hidden
        self.v = nn.Linear(hidden_dim, 1, bias=False)            # 压成标量分数

    def forward(self, dec_hidden, enc_outputs, src_mask):
        # dec_hidden:  (1, B, H) -> 取出 (B, H)
        # enc_outputs: (B, T_src, H)
        # src_mask:    (B, T_src)  bool, True 表示真实 token
        s = dec_hidden.squeeze(0)                                # (B, H)
        s = s.unsqueeze(1)                                       # (B, 1, H) 广播到每个 j
        # W·h + U·s ：靠广播相加
        scores = self.v(torch.tanh(self.W(enc_outputs) + self.U(s))).squeeze(-1)  # (B, T_src)
        scores = scores.masked_fill(~src_mask, float('-inf'))
        alpha  = F.softmax(scores, dim=-1)                       # (B, T_src)
        context = torch.bmm(alpha.unsqueeze(1), enc_outputs).squeeze(1)  # (B, H)
        return context, alpha


### 7.4 Decoder：每步算 attention，再喂 GRU


In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.attention = BahdanauAttention(hidden_dim)
        self.gru = nn.GRU(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc  = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden, enc_outputs, src_mask):
        # x:           (B, 1)
        # hidden:      (1, B, H)
        # enc_outputs: (B, T_src, H)
        # src_mask:    (B, T_src)
        embedded = self.embedding(x)                                # (B, 1, E)
        context, alpha = self.attention(hidden, enc_outputs, src_mask)  # context: (B, H)
        gru_in = torch.cat([embedded, context.unsqueeze(1)], dim=-1)    # (B, 1, E+H)
        output, hidden = self.gru(gru_in, hidden)                       # output: (B, 1, H)
        logits = self.fc(output.squeeze(1))                              # (B, V)
        return logits, hidden, alpha


### 7.5 Seq2Seq：透传 mask、收集 alpha


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, src_len, tgt, teacher_forcing_ratio=0.5):
        B, T_tgt = tgt.size()
        V = self.decoder.fc.out_features
        outputs = torch.zeros(B, T_tgt, V, device=src.device)

        enc_outputs, hidden = self.encoder(src, src_len)
        src_mask = (src != PAD)                          # (B, T_src)

        input_tok = tgt[:, 0:1]                          # (B, 1) SOS
        for t in range(1, T_tgt):
            logits, hidden, _ = self.decoder(input_tok, hidden, enc_outputs, src_mask)
            outputs[:, t, :] = logits
            use_teacher = random.random() < teacher_forcing_ratio
            input_tok = tgt[:, t:t+1] if use_teacher else logits.argmax(-1, keepdim=True)
        return outputs


### 7.6 训练循环（与 U10 几乎相同）


In [ ]:
encoder = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM)
decoder = Decoder(len(tgt_vocab), EMBED_DIM, HIDDEN_DIM)
model   = Seq2Seq(encoder, decoder)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn   = nn.CrossEntropyLoss(ignore_index=PAD)

V_tgt = len(tgt_vocab)
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, n = 0.0, 0
    for src, src_len, tgt in loader:
        optimizer.zero_grad()
        outputs = model(src, src_len, tgt, teacher_forcing_ratio=0.5)
        loss = loss_fn(
            outputs[:, 1:, :].reshape(-1, V_tgt),
            tgt[:, 1:].reshape(-1),
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item(); n += 1
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | loss={total_loss/n:.4f}')


### 7.7 推理 + 收集 attention 权重


In [ ]:
@torch.no_grad()
def translate(model, sentence, max_len=20):
    model.eval()
    src_ids = sentence_to_ids(sentence, src_vocab, tokenize_zh, add_sos=False, add_eos=True)
    src     = torch.tensor([src_ids], dtype=torch.long)
    src_len = torch.tensor([len(src_ids)], dtype=torch.long)

    enc_outputs, hidden = model.encoder(src, src_len)
    src_mask = (src != PAD)

    input_tok = torch.tensor([[SOS]], dtype=torch.long)
    out_ids, alphas = [], []
    for _ in range(max_len):
        logits, hidden, alpha = model.decoder(input_tok, hidden, enc_outputs, src_mask)
        next_id = logits.argmax(-1).item()
        if next_id == EOS:
            break
        out_ids.append(next_id)
        alphas.append(alpha.squeeze(0).cpu().numpy())   # (T_src,)
        input_tok = torch.tensor([[next_id]], dtype=torch.long)
    return tgt_vocab.decode(out_ids), src_ids, alphas

# 看几条样本
for zh, en in raw_pairs[:5]:
    pred, _, _ = translate(model, zh)
    print(f'{zh}  ->  {" ".join(pred)}   (gold: {en})')


## 8. 注意力热力图

每生成一个目标词，我们都拿到了一个 `(T_src,)` 的权重向量。把它们沿时间步堆起来，就是一张 `(T_tgt, T_src)` 的矩阵——这就是热力图。

观察重点：
- 对角线趋势：源句和目标句近似按顺序对齐
- 高亮块：模型关注到了对应的源词（如生成 `you` 时高亮「你」）
- 散乱：模型还没学好对齐，或者数据太少


In [ ]:
import numpy as np

def show_attention(zh):
    pred, src_ids, alphas = translate(model, zh)
    src_tokens = src_vocab.decode(src_ids)
    matrix = np.stack(alphas, axis=0)   # (T_tgt, T_src)

    try:
        import matplotlib.pyplot as plt
        import matplotlib
        # 使用系统中可用的 CJK 字体；找不到也不影响热力图本身
        for f in ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'PingFang SC']:
            try:
                matplotlib.rcParams['font.sans-serif'] = [f]
                matplotlib.rcParams['axes.unicode_minus'] = False
                break
            except Exception:
                pass
        fig, ax = plt.subplots(figsize=(max(4, len(src_tokens)*0.6),
                                         max(3, len(pred)*0.5)))
        im = ax.imshow(matrix, aspect='auto', cmap='viridis')
        ax.set_xticks(range(len(src_tokens))); ax.set_xticklabels(src_tokens, rotation=0)
        ax.set_yticks(range(len(pred)));        ax.set_yticklabels(pred)
        ax.set_xlabel('source'); ax.set_ylabel('target')
        plt.colorbar(im, ax=ax)
        plt.tight_layout(); plt.show()
    except ImportError:
        # 没有 matplotlib 就退化为文本输出
        print('source :', src_tokens)
        for i, tok in enumerate(pred):
            row = ' '.join(f'{x:.2f}' for x in matrix[i])
            print(f'{tok:>10s} | {row}')

show_attention('我爱你')
show_attention('她正在学习深度学习')


## 9. Attention vs Baseline 对比

| 维度 | U10 Baseline | U11 Attention |
|---|---|---|
| Encoder 输出 | `hidden (1,B,H)` | `outputs (B,T_src,H)` + `hidden` |
| context 是否随时间变 | 否，所有 t 共享 | 是，每个 t 都重新算 |
| Decoder GRU 输入 | `(B, 1, E)` | `(B, 1, E+H)` |
| 是否需要 src_mask | 否 | 是（pad 位置 score 设 -inf） |
| 是否能可视化对齐 | 否 | 是（α 矩阵就是热力图） |
| 长句性能 | 显著下降 | 明显改善 |

**计算量**：每个 decoder step 多了 `O(T_src · H)` 的 attention 计算。在长源句下不可忽视——这正是 Transformer 用并行化和点积 attention 的动机。


## 10. 本单元小结

- **核心改造**：Decoder 不再共享一个 context，而是每步现算 `c_t = Σ α_tj · h_j`
- **四步公式**：score（v·tanh(W·h+U·s)） → softmax → 加权求和 → 与 embedding 拼接喂 GRU
- **三处改动**：Encoder 多返回 outputs；Decoder 加 attention 模块、GRU 输入维度变成 E+H；Seq2Seq 透传 src_mask
- **mask 关键**：pad 位置 score 设 `-inf`，softmax 后权重精确为 0
- **副产品**：α 矩阵直接画热力图，是 Attention 类模型最直观的可解释性来源
- **历史脉络**：Bahdanau（加性，2014）→ Luong（乘性，2015）→ Transformer（scaled dot-product，2017）

### 必须能默写

1. Bahdanau attention 的 score 公式
2. attention mask 为什么设 -inf 而不是 0
3. Decoder GRU 的输入维度从 E 变成 E+H 的原因
4. 注意力权重 α 的形状和含义
5. 为什么 Encoder 现在要返回 outputs 而不只是 hidden
